# LaTeX Preparation

## Main Results

In [ ]:
import numpy as np
import os
import pandas as pd

from constants import (
    OBJECTIVE_FUNCTIONS_NAMES,
    ACQ_TYPE_MAPPING,
    ALGO_FILE_COUNT,
    LLMGP_NUMERICAL_RESULTS_DIR,
    NUMERICAL_RESULTS_DIR,
    EXP_RUNS,
)

def read_raw_result(problem, acq_type, result_type):
    raw_result = []
    for exp_idx in range(EXP_RUNS):
        try:
            if acq_type == "llmgp":
                file_name = f"{LLMGP_NUMERICAL_RESULTS_DIR}/{problem}/llmgp/{exp_idx}_{result_type}.npy"
            else:
                file_name = f"{NUMERICAL_RESULTS_DIR}/{problem}/{acq_type}/{exp_idx}_{result_type}.npy"
            result_sequence = np.load(file_name)
            if np.isnan(result_sequence).any():
                print(f"Found nan value in {file_name}")
                os.remove(file_name)  # Remove the file if it contains NaN values
                continue
            # if result_type is "simple_regret", check if any number is negative and print file name:
            if result_type == "simple_regret" and any(result < 0 for result in result_sequence):
                print(f"Found negative value in {file_name}")
                return []
            raw_result.append(result_sequence)
        except FileNotFoundError:
            continue 
    return raw_result

def get_agg_result(raw_result, agg):
    if agg == "auc":
        agg_result = np.trapezoid(raw_result.squeeze(), dx=1.0).item()
    elif agg == "mean":
        agg_result = np.mean(raw_result)
    elif agg == "last":
        agg_result = raw_result[-1]
    return agg_result

def get_all_problem_raw_result(problem_list, method_list, result_type):
    problem_result = {}
    for problem in problem_list:
        problem_result[problem] = {}
        for method in method_list:
            problem_result[problem][method] = read_raw_result(problem, method, result_type)
    return problem_result

def find_best_result_per_problem(result_by_all_methods):
    # set best to infty
    best_result = float("inf")
    for _, results in result_by_all_methods.items():
        for result in results:
            if result < best_result:
                best_result = result
    return best_result

def load_results_and_empirical_performance(problem_list, method_list, result_type):
    all_raw_results = get_all_problem_raw_result(problem_list, method_list, result_type)
    empirical_optimum = {}
    for problem, problem_raw_results in all_raw_results.items():
        minimum_value = float("inf")
        for method_raw_results in problem_raw_results.values():
            method_minimum_value = [min(result_sequence).item() for result_sequence in method_raw_results]
            if len(method_minimum_value)==0:
                continue
            elif minimum_value > min(method_minimum_value):
                minimum_value = min(method_minimum_value)
        empirical_optimum[problem] = minimum_value
    return all_raw_results, empirical_optimum

def cal_simple_regret(all_raw_results, empirical_optimum):
    all_simple_regrets = {}

    for problem, methods in all_raw_results.items():
        all_simple_regrets[problem] = {}
        optimum = empirical_optimum[problem]
        for method, runs in methods.items():
            all_simple_regrets[problem][method] = []
            for i, run in enumerate(runs):
                # run is a numpy array of values for all iterations
                # first filter the actual run
                if run.shape[0] > 100:
                    run = run[-101:]
                else:
                    run = run[-51:]
                # then get the current best value at each iteration
                best_values = np.minimum.accumulate(run)
                # get the simple regret
                simple_regret = best_values - optimum
                all_simple_regrets[problem][method].append(simple_regret)
                assert len(simple_regret) == 101 or len(simple_regret) == 51
                assert np.all(simple_regret >= 0), f"Negative simple regret found at {problem}-{method}{i}"
    return all_simple_regrets

def aggregate_and_to_df(all_simple_regrets, agg):
    # Aggregate by AUC for each run
    mean_simple_regrets = {}
    for problem, methods in all_simple_regrets.items():
        mean_simple_regrets[problem] = {}
        for method, runs in methods.items():
            mean_simple_regrets[problem][method] = []
            for run in runs:
                mean_simple_regrets[problem][method].append(get_agg_result(run, agg))
    # Create a DataFrame from mean_simple_regrets
    agg_simple_regrets_df = pd.DataFrame([
        {"problem": problem, "method": method, **{f"run_{i+1}": run_mean for i, run_mean in enumerate(run_means)}}
        for problem, methods in mean_simple_regrets.items()
        for method, run_means in methods.items()
    ])
    return agg_simple_regrets_df

def get_relative_performance(agg_simple_regrets_df):
    # get the sum across all runs
    rel_performance_df = agg_simple_regrets_df.copy()
    rel_performance_df["sum"] = rel_performance_df.iloc[:, 2:].sum(axis=1)
    # for each problem, divide the sum by the best sum
    for problem in rel_performance_df["problem"].unique():
        best_sum = rel_performance_df.loc[rel_performance_df["problem"] == problem, "sum"].min()
        rel_performance_df.loc[rel_performance_df["problem"] == problem, "relative_performance"] = rel_performance_df["sum"] / best_sum
    return rel_performance_df

def summary_by_method(rel_performance_df):
    # Summarize the relative performance by method
    summary_df = rel_performance_df.groupby("method")["relative_performance"].agg(["median", "min", "max"]).reset_index()
    summary_df["range"] = summary_df["max"] - summary_df["min"]
    return summary_df

In [27]:
all_methods = list(ACQ_TYPE_MAPPING.keys())
all_methods.extend(list(ALGO_FILE_COUNT.keys()))
all_problems = OBJECTIVE_FUNCTIONS_NAMES

all_raw_results, empirical_optimum = load_results_and_empirical_performance(all_problems, all_methods, "train_Y")
all_simple_regrets = cal_simple_regret(all_raw_results, empirical_optimum)
agg_simple_regrets_df = aggregate_and_to_df(all_simple_regrets, "auc")
rel_performance_df = get_relative_performance(agg_simple_regrets_df)
summary_df = summary_by_method(rel_performance_df)

In [29]:
rel_performance_df

,problem,method,run_1,run_2,run_3,run_4,run_5,run_6,run_7,run_8,run_9,run_10,sum,relative_performance
0,Ackley,PI,2014.671588,2026.387295,2024.643217,2038.939139,1992.427175,1988.847380,1986.440362,2024.278719,2015.441765,1994.769315,20106.845956,1.117876
1,Ackley,LogPI,1836.894743,1856.657441,1849.739451,1783.282186,1791.548346,1806.463962,1823.329097,1925.055259,1888.859528,1880.179836,18442.009848,1.025317
2,Ackley,EI,2031.505977,2054.290774,2064.725796,2030.036484,2008.102637,1675.216190,2055.162968,2023.419705,2012.280120,2025.419193,19980.159844,1.110833
3,Ackley,LogEI,1669.412909,1803.339326,1913.854864,1864.061438,1757.685067,1714.501165,1829.329003,1839.324949,1901.654489,1700.579923,17993.743132,1.000394
4,Ackley,UCB,2080.178554,2079.172389,2073.869987,2058.930586,2075.388580,2082.443303,2064.109581,2084.786800,2083.887659,2084.911206,20767.678645,1.154616
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1615,hpt_diabetes_MLPSGD,llambo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN
1616,hpt_diabetes_MLPSGD,bo_alternating_k1,93.101399,93.327535,105.367237,99.810662,96.535218,99.250733,98.962103,102.017781,98.729978,99.261044,986.363691,inf
1617,hpt_diabetes_MLPSGD,bo_alternating_k3,103.247109,96.383857,93.687541,102.812447,94.506855,105.170917,100.431105,99.647559,101.836845,92.980240,990.704474,inf
1618,hpt_diabetes_MLPSGD,bo_alternating_k5,103.125493,92.760042,103.544444,94.448505,94.754395,95.838479,107.357122,104.314868,92.128614,110.646886,998.918849,inf


In [10]:
# Aggregate by AUC for each run
mean_simple_regrets = {}
for problem, methods in all_simple_regrets.items():
    mean_simple_regrets[problem] = {}
    for method, runs in methods.items():
        mean_simple_regrets[problem][method] = []
        for run in runs:
            mean_simple_regrets[problem][method].append(get_agg_result(run, "auc"))
# Create a DataFrame from mean_simple_regrets
mean_simple_regrets_df = pd.DataFrame([
    {"problem": problem, "method": method, **{f"run_{i+1}": run_mean for i, run_mean in enumerate(run_means)}}
    for problem, methods in mean_simple_regrets.items()
    for method, run_means in methods.items()
])
# Find the best simple regret across all runs by all methods for each problem
# Melt the DataFrame to long format for easier min selection
melted_df = mean_simple_regrets_df.melt(id_vars=["problem", "method"], var_name="run", value_name="mean_simple_regret")
best_simple_regrets = melted_df.loc[melted_df.groupby("problem")["mean_simple_regret"].idxmin()]
# Get the relative performance compared to the best simple regret for each problem
# Add a column for the best simple regret per problem
melted_df = melted_df.merge(
    best_simple_regrets[["problem", "mean_simple_regret"]].rename(columns={"mean_simple_regret": "best_simple_regret"}),
    on="problem",
    how="left"
)
# Calculate relative performance
melted_df["relative_performance"] = melted_df["mean_simple_regret"] / melted_df["best_simple_regret"]
# Aggregate by problem and method
# Take the median, min, and max of the mean_simple_regret across all runs
all_results = melted_df.groupby(["problem", "method"])["relative_performance"].agg(["median", "min", "max"]).reset_index()
all_results["range"] = all_results["median"] - all_results["min"]

In [18]:
temp_df = mean_simple_regrets_df[mean_simple_regrets_df['problem']=='Ackley']
temp_df["average"] = temp_df[["run_1", "run_2", "run_3", "run_4", "run_5", "run_6", "run_7", "run_8", "run_9", "run_10"]].mean(axis=1)

/tmp/ipykernel_2347297/3469932125.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["average"] = temp_df[["run_1", "run_2", "run_3", "run_4", "run_5", "run_6", "run_7", "run_8", "run_9", "run_10"]].mean(axis=1)


In [19]:
temp_df

,problem,method,run_1,run_2,run_3,run_4,run_5,run_6,run_7,run_8,run_9,run_10,average
0,Ackley,PI,2014.671588,2026.387295,2024.643217,2038.939139,1992.427175,1988.847380,1986.440362,2024.278719,2015.441765,1994.769315,2010.684596
1,Ackley,LogPI,1836.894743,1856.657441,1849.739451,1783.282186,1791.548346,1806.463962,1823.329097,1925.055259,1888.859528,1880.179836,1844.200985
2,Ackley,EI,2031.505977,2054.290774,2064.725796,2030.036484,2008.102637,1675.216190,2055.162968,2023.419705,2012.280120,2025.419193,1998.015984
3,Ackley,LogEI,1669.412909,1803.339326,1913.854864,1864.061438,1757.685067,1714.501165,1829.329003,1839.324949,1901.654489,1700.579923,1799.374313
4,Ackley,UCB,2080.178554,2079.172389,2073.869987,2058.930586,2075.388580,2082.443303,2064.109581,2084.786800,2083.887659,2084.911206,2076.767864
5,Ackley,PosMean,1907.567658,1952.006368,1987.966460,1953.616740,1889.197142,1872.464936,1882.749861,1987.052772,1945.066946,2001.179179,1937.886806
6,Ackley,PosSTD,1909.948872,2006.299621,1965.673157,1989.775475,1847.884234,1977.701176,1911.764448,1910.541504,1961.861990,1905.734165,1938.718464
7,Ackley,TS,2082.848814,2066.510793,2083.843498,2047.512516,2085.816493,2083.589925,2072.248274,2075.859303,2089.188161,2073.068207,2076.048598
8,Ackley,qKG,2011.690812,1954.605497,2026.791260,2027.732960,2035.963295,1990.857362,2003.422864,2010.401637,2018.438086,1983.477865,2006.338164
9,Ackley,qPES,2100.062435,2103.816790,2092.296082,2092.680198,2096.654057,2100.846108,2075.765441,2099.287166,2082.436532,2090.453205,2093.429801


In [15]:
all_results.head()

,problem,method,median,min,max,range
0,Ackley,EI,5.260252,4.345780,5.356230,0.914472
1,Ackley,LogEI,4.711862,4.330725,4.964847,0.381137
2,Ackley,LogPI,4.781860,4.626120,4.993902,0.155740
3,Ackley,PI,5.227381,5.153145,5.289335,0.074235
4,Ackley,PosMean,5.054817,4.857475,5.191380,0.197342


In [13]:
# Ignore 5 hpt_diabetes problems for now
hpt_diabetes_problems = [
    "hpt_diabetes_RandomForest",
    "hpt_diabetes_DecisionTree",
    "hpt_diabetes_SVM",
    "hpt_diabetes_AdaBoost",
    "hpt_diabetes_MLPSGD"
]
## Remove them from the dataframe
all_results = all_results[~all_results['problem'].isin(hpt_diabetes_problems)]
# Select the results of the first 40 problems in OBJECTIVE_FUNCTION_NAMES
all_results = all_results[all_results['problem'].isin(OBJECTIVE_FUNCTIONS_NAMES[:40])]

# On each problem, get the rank of the methods
problem_ranks = {}
for problem in all_results['problem'].unique():
    problem_data = all_results[all_results['problem'] == problem]
    # Rank methods by median relative performance (1 = best, higher = worse)
    problem_data_ranked = problem_data.sort_values('median').reset_index(drop=True)
    problem_data_ranked['rank'] = range(1, len(problem_data_ranked) + 1)
    problem_ranks[problem] = problem_data_ranked[['method', 'rank']].set_index('method')['rank'].to_dict()

# Create a DataFrame with methods as columns and problems as rows, values are ranks
rank_df = pd.DataFrame(problem_ranks).T.fillna(len(all_results['method'].unique()) + 1)
# Calculate average, min, and max rank for each method
rank_summary = rank_df.agg(['mean', 'min', 'max']).T
rank_summary.columns = ['avg_rank', 'min_rank', 'max_rank']
rank_summary = rank_summary.sort_values('avg_rank')


In [17]:
problem_ranks["Ackley"]

{'LogEI': 1,
 'LogPI': 2,
 'esp': 3,
 'lmabo-ab2': 4,
 'lmabo-ab1': 5,
 'lmabo': 6,
 'PosSTD': 7,
 'gphedge': 8,
 'lmabo-ab3': 9,
 'PosMean': 10,
 'setup_bo': 11,
 'qMES': 12,
 'lmabo-ops': 13,
 'llambo': 14,
 'qKG': 15,
 'PI': 16,
 'bo_alternating_k3': 17,
 'EI': 18,
 'bo_alternating_k1': 19,
 'bo_alternating_k5': 20,
 'qJES': 21,
 'bo_explore_exploit': 22,
 'TS': 23,
 'UCB': 24,
 'no_past_bo': 25,
 'llmgp': 26,
 'qPES': 27}

In [14]:
rank_summary

,avg_rank,min_rank,max_rank
LogPI,5.750,1.0,24.0
PosMean,7.175,1.0,25.0
PI,7.650,1.0,26.0
lmabo,8.375,1.0,19.0
LogEI,8.875,1.0,22.0
EI,9.475,1.0,24.0
gphedge,9.775,1.0,22.0
lmabo-ab2,10.825,2.0,21.0
bo_alternating_k1,11.450,4.0,20.0
bo_alternating_k5,12.225,1.0,23.0


# AF Frequency

Show the overall selection of AFs by LMABO.

## Overall frequency

In [ ]:
from constants import OBJECTIVE_FUNCTIONS_NAMES

def load_af_choices(problem):
    folder_path = f"numerical_results/{problem}/lmabo"
    af_choices = {}
    for i in range(10):
        file_path = f"{folder_path}/{i}_acq_types.txt"
        with open(file_path, 'r') as file:
            # each AF sits in a line
            af_choices[i] = [af for af in file.read().splitlines() if af]
    return af_choices

def load_all_af_choices():
    all_af_choices = {}
    for problem in OBJECTIVE_FUNCTIONS_NAMES:
        all_af_choices[problem] = load_af_choices(problem)
    return all_af_choices

In [ ]:
all_af_choices = load_all_af_choices()
# count all AFs
af_count = {}
for problem, problem_choices in all_af_choices.items():
    for i, afs in problem_choices.items():
        for af in afs:
            if af not in af_count:
                af_count[af] = 0
            af_count[af] += 1
# plot the sorted AF count by a bar chart
import matplotlib.pyplot as plt
af_count = dict(sorted(af_count.items(), key=lambda item: item[1], reverse=True))
plt.bar(af_count.keys(), af_count.values())
plt.xticks(rotation=90)
plt.xlabel('Acquisition Function')
plt.ylabel('Count')
plt.title('Acquisition Function Count')
plt.tight_layout()
plt.show()

## Dataset-specific frequency

In [ ]:
# Plot the AF choices for each problem by a heatmap
# For each problem, count the number of choices for each AF
# Create a dataframe where rows are problems and columns are AF choices
import seaborn as sns
import numpy as np
import pandas as pd
problems = list(all_af_choices.keys())
af_count_by_problem = {problem: {af: 0 for af in af_count.keys()} for problem in problems}
for problem, choices in all_af_choices.items():
    for i, afs in choices.items():
        for af in afs:
            af_count_by_problem[problem][af] += 1
af_df = pd.DataFrame(af_count_by_problem).T.fillna(0)
af_df = af_df[sorted(af_df.columns)]
plt.figure(figsize=(12, 8))
sns.heatmap(af_df, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={'label': 'Count'})
plt.xlabel('Acquisition Function')
plt.ylabel('Problem')
plt.title('Acquisition Function Choices by Problem')
plt.tight_layout()
plt.show()

# Temporal Analysis

## AF choice by iteration

In [ ]:
# Count the frequency of each AF per iteration across all problems
# We either have 50 or 100 iterations, depending on the problem, so we will group by 50
# Create a dataframe where rows are iterations and columns are AF choices
# For problems with 100 iterations, we will group them by 50, which means iterations 1 and 2 will be grouped together, 3 and 4, etc.
af_count_by_iteration = {i: {af: 0 for af in af_count.keys()} for i in range(1, 51)}
for problem, choices in all_af_choices.items():
    for afs in choices.values():
        for j, af in enumerate(afs):
            # Group by 50 iterations
            if len(afs) == 50:
                iteration = j + 1
            else:
                # For 100 iterations, we group them by 50
                if j % 2 == 0:  # even index means first of the pair
                    iteration = (j // 2) + 1
            af_count_by_iteration[iteration][af] += 1
af_iter_df = pd.DataFrame(af_count_by_iteration).T.fillna(0)
af_iter_df = af_iter_df[sorted(af_iter_df.columns)]
plt.figure(figsize=(12, 8))
sns.heatmap(af_iter_df, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={'label': 'Count'})
plt.xlabel('Acquisition Function')
plt.ylabel('Iteration')
plt.title('Acquisition Function Choices by Iteration')
plt.tight_layout()
plt.show()

## AF choice with switches

In [ ]:
# a heatmap of the number of times the algorithm switched from one AF to another
# For each problem, count the number of switches from one AF to another across all 10 runs
af_switch_count = {af: {af2: 0 for af2 in af_count.keys()} for af in af_count.keys()}
for problem, choices in all_af_choices.items():
    for afs in choices.values():
        for j in range(len(afs) - 1):
            af1 = afs[j]
            af2 = afs[j + 1]
            if af1 != af2:
                af_switch_count[af1][af2] += 1
af_switch_df = pd.DataFrame(af_switch_count).fillna(0)
af_switch_df = af_switch_df[sorted(af_switch_df.columns)]
plt.figure(figsize=(12, 8))
sns.heatmap(af_switch_df, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={'label': 'Switch Count'})
plt.xlabel('From Acquisition Function')
plt.ylabel('To Acquisition Function')
plt.title('Acquisition Function Switch Count')
plt.tight_layout()
plt.show()

In [ ]:
# I want to check how often does the algorithm switch from one AF to another in all_af_choices.
# Make separate counts for runs with 50 iterations and 100 iterations.

switch_counts = {
    "50": [],
    "100": []
}

for all_choices_by_problem in all_af_choices.values():
    for choices in all_choices_by_problem.values():
        switch_count = 0
        for i in range(len(choices) - 1):
            if choices[i] != choices[i + 1]:
                switch_count += 1
        if len(choices) == 50:
            switch_counts["50"].append(switch_count)
        else:
            switch_counts["100"].append(switch_count)
# Now present the two as boxplots
import matplotlib.pyplot as plt

plt.boxplot([switch_counts["50"], switch_counts["100"]], labels=["50 Iterations", "100 Iterations"])
plt.ylabel("Number of Switches")
plt.title("Switch Counts for Different Iteration Counts")
plt.show()

In [ ]:
avg_50 = sum(switch_counts["50"]) / len(switch_counts["50"])
avg_100 = sum(switch_counts["100"]) / len(switch_counts["100"])
print(f"Average switch count for 50 iterations: {avg_50:.2f}")
print(f"Average switch count for 100 iterations: {avg_100:.2f}")

# State-conditioned Analysis

## Stagnation vs Improvement

In [ ]:
# first define functions to load the simple regret of LMABO, modified from load_af_choices and load_all_af_choices

from constants import OBJECTIVE_FUNCTIONS_NAMES

def load_simple_regret(problem):
    folder_path = f"numerical_results/{problem}/lmabo"
    simple_regret = {}
    for i in range(10):
        file_path = f"{folder_path}/{i}_simple_regret.npy"
        simple_regret[i] = np.load(file_path)
    return simple_regret

def load_all_simple_regret():
    all_simple_regret = {}
    for problem in OBJECTIVE_FUNCTIONS_NAMES:
        all_simple_regret[problem] = load_simple_regret(problem)
    return all_simple_regret


In [ ]:
# Load all simple regret data
all_simple_regret = load_all_simple_regret()
# Define stagnation and improvement criteria
threshold = 1e-5
# Convert simple regret to stagnation or improvement
# True (improvement) if the simple regret reduces more than the threshold
# False (stagnation) if the simple regret does not reduce more than the threshold
# Do this for each array where an array of N iterations is converted to a list of N binary labels
all_simple_regret_status = {}
for problem in all_simple_regret.keys():
    all_simple_regret_status[problem] = {}
    for i, simple_regret in all_simple_regret[problem].items():
        # Convert to stagnation or improvement
        status = [(simple_regret[j] - simple_regret[j + 1]).item() > threshold for j in range(len(simple_regret) - 1)]
        all_simple_regret_status[problem][i] = status

In [ ]:
# Now we combine all_simple_regret_status and all_af_choices to count the number of stagnations and improvements for each AF type.
af_combined_status = {af: {"Stagnation": 0, "Improvement": 0} for af in af_count.keys()}
for problem, simple_regret_status_by_problem in all_simple_regret_status.items():
    af_choices_by_problem = all_af_choices[problem]
    for i, status_series in simple_regret_status_by_problem.items():
        af_series = af_choices_by_problem[i]
        for j, status in enumerate(status_series):
                af = af_series[j]
                # Count stagnation and improvement
                if status:
                    af_combined_status[af]["Improvement"] = af_combined_status[af].get("Improvement", 0) + 1
                else:
                    af_combined_status[af]["Stagnation"] = af_combined_status[af].get("Stagnation", 0) + 1

# Create a dataframe for heatmap: rows are AFs, columns are ["Stagnation", "Improvement"], values are proportions across the columns
af_status_df = pd.DataFrame(af_combined_status).T
improvement_total = af_status_df["Improvement"].sum()
stagnation_total = af_status_df["Stagnation"].sum()
af_status_df["Stagnation_prop"] = af_status_df["Stagnation"] / stagnation_total
af_status_df["Improvement_prop"] = af_status_df["Improvement"] / improvement_total
heatmap_df = af_status_df[["Stagnation_prop", "Improvement_prop"]]
heatmap_df = heatmap_df.rename(columns={"Stagnation_prop": "Stagnation", "Improvement_prop": "Improvement"})

plt.figure(figsize=(8, len(heatmap_df) * 0.4))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="coolwarm", cbar_kws={'label': 'Proportion'})
plt.ylabel("Acquisition Function")
plt.title("Proportion of Acquisition Functions for Stagnation vs Improvement")
plt.tight_layout()
plt.show()

## Correlation with GP model parameters

In [ ]:
import re
import pandas as pd

log_path = "log/Rosenbrock_lmabo.out"

def read_params_af_log(log_path):
    # Patterns for extraction
    run_pattern = re.compile(r"RUN (\d+)")
    iter_pattern = re.compile(r"Iter (\d+)")
    lengthscale_pattern = re.compile(
        r"Lengthscales: Range \[([0-9.eE+-]+), ([0-9.eE+-]+)\], Mean ([0-9.eE+-]+) \(Std Dev ([0-9.eE+-]+)\)"
    )
    outputscale_pattern = re.compile(r"Outputscale: ([0-9.eE+-]+)")
    shortest_dist_pattern = re.compile(r"Shortest distance: ([0-9.eE+-]+)")
    llm_af_pattern = re.compile(r"^(EI|TS|UCB|LogEI|qPES|qMES|qJES|qKG|PI|LogPI|PosSTD|PosMean):|LLM suggested AF: ([A-Za-z0-9]+)")

    results = []
    current_run = None
    current_iter = None

    with open(log_path, "r") as f:
        for line in f:
            # Detect run
            run_match = run_pattern.match(line)
            if run_match:
                current_run = int(run_match.group(1))
                continue

            # Detect iteration
            iter_match = iter_pattern.match(line)
            if iter_match:
                current_iter = int(iter_match.group(1))
                continue

            # Extract lengthscale
            lengthscale_match = lengthscale_pattern.search(line)
            if lengthscale_match:
                ls_min = float(lengthscale_match.group(1))
                ls_max = float(lengthscale_match.group(2))
                ls_mean = float(lengthscale_match.group(3))
                ls_std = float(lengthscale_match.group(4))
                continue

            # Extract outputscale
            outputscale_match = outputscale_pattern.search(line)
            if outputscale_match:
                outputscale = float(outputscale_match.group(1))
                continue

            # Extract shortest distance
            shortest_dist_match = shortest_dist_pattern.search(line)
            if shortest_dist_match:
                shortest_dist = float(shortest_dist_match.group(1))
                continue

            # Extract AF
            af_match = llm_af_pattern.search(line)
            if af_match:
                af = af_match.group(1) if af_match.group(1) else af_match.group(2)
                # Save only if all fields are present
                if current_run is not None and current_iter is not None:
                    results.append({
                        "run": current_run,
                        "iteration": current_iter,
                        "lengthscale_min": ls_min,
                        "lengthscale_max": ls_max,
                        "lengthscale_mean": ls_mean,
                        "lengthscale_std": ls_std,
                        "outputscale": outputscale,
                        "shortest_distance": shortest_dist,
                        "af": af
                    })
    return pd.DataFrame(results)

# Example: print first 5 entries
params_af_df = read_params_af_log(log_path)
print(params_af_df.shape)
params_af_df.head()

In [ ]:
# Summarize the frequence of af
af_frequency = params_af_df['af'].value_counts()
print("Frequency of Acquisition Functions:")
print(af_frequency)

In [ ]:
import matplotlib.pyplot as plt

numerical_cols = ['lengthscale_min', 'lengthscale_max', 'lengthscale_mean', 'lengthscale_std', 'outputscale', 'shortest_distance']

avg_by_iter = params_af_df.groupby('iteration')[numerical_cols].mean()

fig, axes = plt.subplots(len(numerical_cols), 1, figsize=(10, 3 * len(numerical_cols)), sharex=True)
for idx, col in enumerate(numerical_cols):
    axes[idx].plot(avg_by_iter.index, avg_by_iter[col], marker='o')
    axes[idx].set_ylabel(col)
    axes[idx].grid(True)
axes[-1].set_xlabel('Iteration')
fig.suptitle('Average GP Parameter Values per Iteration Across All Runs', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# List of numerical columns to plot
numerical_cols = ['lengthscale_min', 'lengthscale_max', 'lengthscale_mean', 'lengthscale_std', 'outputscale', 'shortest_distance']

# Create a boxplot for each parameter grouped by acquisition function
fig, axes = plt.subplots(len(numerical_cols), 1, figsize=(12, 4 * len(numerical_cols)), sharex=True)
for idx, col in enumerate(numerical_cols):
    # convert lengthscale_max, lengthscale_mean, lengthscale_std to log scale
    if col in ['lengthscale_max', 'lengthscale_mean', 'lengthscale_std']:
        params_af_df[col] = params_af_df[col].apply(lambda x: np.log(x) if x > 0 else 0)
    params_af_df.boxplot(column=col, by='af', ax=axes[idx], grid=False, showfliers=False)
    axes[idx].set_title(f'{col} by Acquisition Function')
    # we need to show the acquisition function names for xticks for all boxplots
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=90)
    axes[idx].set_xlabel('Acquisition Function')
    axes[idx].set_ylabel(col)
    # axes[idx].tick_params(axis='x', rotation=90)
plt.suptitle('GP Parameters Grouped by Acquisition Function', fontsize=16)
plt.show()

In [ ]:
# aggregate the numerical columns by af
agg_params_af_df = params_af_df.groupby('af')[numerical_cols].agg(['mean']).reset_index()
agg_params_af_df.columns = ['af'] + [f"{col}_{stat}" for col, stat in agg_params_af_df.columns[1:]]
agg_params_af_df

In [ ]:
explorative_group = ['UCB', 'TS', 'qKG', 'qMES', 'qPES', 'qJES', 'PosSTD']
exploitative_group = ['EI', 'LogEI', 'PI', 'LogPI', 'PosMean']
# get the summary statistics for each group by taking the mean of the columns
explorative_df = agg_params_af_df[agg_params_af_df['af'].isin(explorative_group)]
print("Explorative")
print(explorative_df.mean(axis=0, numeric_only=True))
exploitative_df = agg_params_af_df[agg_params_af_df['af'].isin(exploitative_group)]
print("Exploitative")
print(exploitative_df.mean(axis=0, numeric_only=True))


In [ ]:
from scipy import stats

explorative_group = ['UCB', 'TS', 'qKG', 'qMES', 'qPES', 'qJES', 'PosSTD']
exploitative_group = ['EI', 'LogEI', 'PI', 'LogPI', 'PosMean']

def stat_test(group1, group2):
    # perform Mann-Whitney U test
    _, p = stats.mannwhitneyu(group1, group2)
    return p < 0.05

def stat_test_by_problem(problem):
    # for each column we want different things
    # lengthscale_min: test if explorative is smaller
    # lengthscale_max: test if explorative is bigger
    # lengthscale_mean: test if explorative is bigger
    # lengthscale_std: test if explorative is bigger
    # outputscale: test if explorative is bigger
    # shortest_distance: test if explorative is bigger

    col_names = ['lengthscale_min', 'lengthscale_max', 'lengthscale_mean', 'lengthscale_std', 'outputscale', 'shortest_distance']
    tests = ['less', 'greater', 'greater', 'greater', 'greater', 'greater']

    log_path = f"log/{problem}_lmabo.out"
    params_af_df = read_params_af_log(log_path)
    # do not run if empty
    if params_af_df.empty:
        print(f"No data found for {problem}. Skipping...")
        return None
    params_af_df['group'] = params_af_df['af'].apply(lambda x: 'Explorative' if x in explorative_group else 'Exploitative')
    result = {'problem': problem, 'nrows': params_af_df.shape[0], 'lengthscale_max_mean': params_af_df['lengthscale_max'].mean()}

    for col, test in zip(col_names, tests): 
        group1 = params_af_df[params_af_df['group'] == 'Explorative'][col]
        group2 = params_af_df[params_af_df['group'] == 'Exploitative'][col]
        _, p_value = stats.mannwhitneyu(group1, group2, alternative=test)
        result[col] = p_value.item()
    return result

In [ ]:
from constants import OBJECTIVE_FUNCTIONS_NAMES
# loop over all problems
all_results = []
for problem in OBJECTIVE_FUNCTIONS_NAMES:
    print(f"Processing {problem}...")
    result = stat_test_by_problem(problem)
    if result is not None:
        all_results.append(result)
# convert results to DataFrame
all_results_df = pd.DataFrame(all_results)
all_results_df.set_index('problem', inplace=True)

In [ ]:
# sort dataframe by lengthscale_max_mean
all_results_df.sort_values(by='lengthscale_max_mean', inplace=True)
all_results_df

In [ ]:
# create a new dataframe to from all_results_df to check if each cell is less than 0.05 for the last 5 columns
p_value_df = all_results_df.iloc[:, -5:].applymap(lambda x: x < 0.05)
p_value_df["nrows"] = all_results_df["nrows"]
p_value_df["lengthscale_max_mean"] = all_results_df["lengthscale_max_mean"]
p_value_df

In [ ]:
# sum each column of p_value_df
p_value_sums = p_value_df.sum()
p_value_sums

# Qualitative and Comparative Analysis

## Meta Strategies